# The v3 model, module by module

A short guide to each module in `02_Model/v3/src`: what it does, the few equations that matter, and a small runnable example.

**The supply chain in one line:** mines extract and concentrate ore or brine → a single global refinery converts concentrate into battery-grade lithium (LCE) → a manufacturer turns that into a cathode-product proxy → end use accumulates it. Exogenous IEA demand pulls material through the chain once a year; firms decide what to offer, buy and build; a ledger records every tonne.

| Module | Role |
|---|---|
| `units.py`, `params.py` | Conversion factors, constants and the `Scenario` settings |
| `ledger.py` | Moves mass between stocks and checks conservation |
| `mines.py` | Extraction and concentration; costs; commitment to build |
| `refinery.py` | Converts concentrate into battery-grade lithium |
| `manufacturer.py`, `end_use.py` | Final processing and accumulation in use |
| `markets.py` | Clears the mine-to-refinery market and sets the price |
| `firms.py` | Agents' decisions: offers, purchases, production, investment |
| `model.py` | The yearly step that ties everything together |
| `create_mines.py`, `usgs_benchmarks.py` | Builds the mine dataset from project data |
| `iea_demand.py` | Loads IEA demand scenarios |
| `setup_model.py` | Assembles a model from the dataset |

In [1]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
model_root = next(
    parent / "02_Model"
    for parent in (working_directory, *working_directory.parents)
    if (parent / "02_Model" / "v3" / "src").is_dir()
)
if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

from v3.src import units
from v3.src.end_use import EndUse
from v3.src.firms import FirmAgent
from v3.src.manufacturer import Manufacturer
from v3.src.markets import SupplyOffer, RefineryRequest, clear_refinery_input_market
from v3.src.mines import Mine
from v3.src.model import Model
from v3.src.params import Scenario
from v3.src.refinery import REAGENT_PER_T_OUTPUT, Refinery


def example_mine(asset_id="example", status="operation", li_fraction=0.006, opex=125.0):
    # A hard-rock mine: 1 Mt of ore at 0.6 % Li, 100 kt ore per year.
    return Mine(asset_id, "illustrative", "hard_rock", status, geological_stock_t_material=1e6,
                capacity_t_material_annual=1e5, composition_mass_fractions={"Li": li_fraction},
                mining_recovery_fraction=0.65, opex_usd_per_t_material=opex, capex_usd=4e7, lead_time_years=8)

## Units and settings: `units.py`, `params.py`

All material is tracked as **component masses** (Li and "other material") in tonnes. Lithium is reported either as contained Li or as lithium carbonate equivalent (LCE):

$$1\ \text{t Li} = 5.323\ \text{t LCE}, \qquad 1\ \text{t Li}_2\text{O} = 0.4645\ \text{t Li}.$$

`Scenario` holds every setting an experiment may change: the refiners' maximum price, refining recoveries and costs, the scarcity premium, the investment hurdle, the commitment probability and the capex payback period.

In [2]:
scenario = Scenario()
print(f"t LCE per t Li: {units.LCE_PER_LI}, t Li per t Li2O: {units.LI_PER_LI2O:.4f}")
scenario

t LCE per t Li: 5.323, t Li per t Li2O: 0.4646


Scenario(mine_markup_fraction=0.0, refinery_required_markup_fraction=0.0, refined_price_per_t=25000.0, scarcity_premium=1.0, investment_hurdle=0.2, commitment_probability=0.5, capex_payback_years=20.0, hard_rock_route=RefiningRoute(lithium_recovery_fraction=0.9, processing_cost_per_t_output=3000.0), brine_route=RefiningRoute(lithium_recovery_fraction=0.85, processing_cost_per_t_output=3500.0))

## Ledger: `ledger.py`

Each asset holds **stocks**: masses of each component in named accounts, such as a mine's geological stock, its concentrate inventory and its residue. Every physical change is an **event** made of **movements** from one account to another. The ledger checks that a withdrawal never exceeds the stock, applies it, and logs it.

For account $a$ and component $c$, a movement of mass $m$ from $a$ to $b$ gives

$$S_{a,c} \leftarrow S_{a,c} - m, \qquad S_{b,c} \leftarrow S_{b,c} + m .$$

Mass is never created or destroyed, so at the end of every year the model checks, for each component,

$$\sum_a S_{a,c}(t) = \sum_a S_{a,c}(0).$$

A failure is a bug, never noise.

In [3]:
mine = example_mine()
model = Model(mines=[mine])
mine.extract(10_000)  # move 10 kt of ore out of the deposit
print("Mass conserved:", model.ledger.check_conservation())
model.ledger.to_dataframes()["movements"][["component", "quantity_t", "source_stock", "destination_stock", "destination_form"]]

Mass conserved: True


,component,quantity_t,source_stock,destination_stock,destination_form
0,Li,39.000000,geological_stock,inventory,spodumene_concentrate
1,other_material,1358.849462,geological_stock,inventory,spodumene_concentrate
2,Li,21.000000,geological_stock,residue,tailings
3,other_material,8581.150538,geological_stock,residue,tailings


**A transfer between assets.** When a mine sells concentrate to the refinery, the ledger records a *transfer*: one movement from the mine's inventory account to the refinery's inventory account for the same material form. The parcel keeps its composition, so each component moves in proportion:

$$m_c = Q \times \frac{S_{\text{mine},c}}{\sum_{c'} S_{\text{mine},c'}}$$

for a parcel of $Q$ tonnes. Nothing changes form in a transfer; only the holder changes. Every trade the market settles (`markets.settle`) is recorded this way, together with its price in `model.transactions`.

In [ ]:
from v3.src.markets import settle

mine = example_mine("mine_A")
refinery = Refinery("refinery", "WORLD", capacity=None)
model = Model(mines=[mine], refinery=refinery)
mine.produce(1_000)  # 1 kt of spodumene concentrate now held at the mine

settle(model, [{"seller_id": "mine_owner", "buyer_id": "refiner", "source_asset_id": "mine_A",
                "receiving_asset_id": "refinery", "material_form": "spodumene_concentrate",
                "quantity_t_input": 400.0, "price_per_t_input": 1_200.0}], event_id="example_trade")

held = lambda asset: sum(asset.stocks[("inventory", "spodumene_concentrate")].values())
print(f"concentrate at the mine: {held(mine):,.0f} t; at the refinery: {held(refinery):,.0f} t")
print("Mass conserved:", model.ledger.check_conservation())
movements = model.ledger.to_dataframes()["movements"]
movements[movements["event_id"].eq("example_trade")][
    ["component", "quantity_t", "source_id", "source_stock", "destination_id", "destination_stock", "destination_form"]]

## Mines: `mines.py`

A mine extracts raw material (ore or brine) and concentrates it to a fixed product grade $G$ (spodumene concentrate: 2.79 % Li, i.e. SC6; concentrated brine: 6 % Li). With feed grade $g$ (Li mass fraction) and mining recovery $r$:

$$\text{yield } y = \frac{g\,r}{G} \quad \text{(t product per t raw material)}$$

For $M$ tonnes extracted, the product holds $g r M$ of lithium plus just enough other material to reach grade $G$; everything else goes to residue.

**Costs.** With operating cost $a$ per tonne of raw material, capex $I$, annual capacity $K$ and payback period $T_p$ (20 years):

$$c = \frac{a}{y}, \qquad c_{\text{full}} = c + \frac{I}{T_p\,K\,y} \quad \text{(USD per t product)}.$$

**Production.** An operating mine can make at most $\min(K, R)\,y$ of product a year ($R$ = remaining raw material). It only produces what the refinery ordered, using held stock first.

**Lifecycle.** pre-construction → (commit) → construction → (after its lead time) → operation.

In [4]:
mine = example_mine()
model = Model(mines=[mine])
print(f"yield: {mine.product_yield:.4f} t product per t ore")
print(f"operating cost: {mine.unit_cost:,.0f} USD per t product; full cost: {mine.full_unit_cost(20):,.0f}")
mine.produce(1_000)  # an order for 1 kt of concentrate
product = mine.stocks[("inventory", "spodumene_concentrate")]
print(f"product: {sum(product.values()):,.0f} t at {product['Li'] / sum(product.values()):.2%} Li")

yield: 0.1398 t product per t ore
operating cost: 894 USD per t product; full cost: 1,037
product: 1,000 t at 2.79% Li


## Refinery: `refinery.py`

One worldwide refinery converts concentrate into battery-grade lithium, measured as LCE. Each input route $r$ has a lithium recovery $R_r$ (hard rock 0.90, brine 0.85) and a processing cost $c_r$ per tonne of output (3,000 and 3,500 USD). Tonnes of LCE per tonne of input:

$$y_r = G\,R_r \times 5.323 .$$

The recovered lithium is combined with reagents into carbonate, so product mass = recovered Li × 5.323. The reagent stock and the capacity cap output:

$$Q_{\max} = \min\!\left(\text{capacity},\ \frac{\text{reagents}}{1 - 1/5.323}\right).$$

Non-recovered lithium and all other feed material go to refining residue. The refinery refines everything it buys within the year.

In [5]:
for form, route in scenario.routes.items():
    grade = 0.0279 if form == "spodumene_concentrate" else 0.06
    print(f"{form}: {grade * route.lithium_recovery_fraction * units.LCE_PER_LI:.3f} t LCE per t input")
print(f"reagent mass per t LCE output: {REAGENT_PER_T_OUTPUT:.3f} t")

spodumene_concentrate: 0.134 t LCE per t input
concentrated_brine: 0.271 t LCE per t input
reagent mass per t LCE output: 0.812 t


## Manufacturer and end use: `manufacturer.py`, `end_use.py`

The manufacturer turns battery-grade lithium into a cathode-product proxy with a fixed efficiency $\eta = 0.95$:

$$\text{product} = \eta \times \text{input}, \qquad \text{residue} = (1-\eta) \times \text{input}.$$

End use takes delivered product into an in-use stock that only grows (no recycling yet).

In [6]:
maker = Manufacturer("manufacturer", "WORLD")
print(f"Input needed for 100 t of product: {100 / maker.efficiency_rate:.1f} t")

Input needed for 100 t of product: 105.3 t


## Markets and price: `markets.py`

Mines offer concentrate; the refinery asks for the refined output it still needs, $Q$. Offers are compared on a common basis, the **equivalent cost per tonne of LCE**:

$$e_i = \frac{p_i}{y_r} + c_r$$

where $p_i$ is the mine's asking price per tonne of product. The cheapest offers are accepted until $Q$ is filled or supply runs out.

**Price.** One price per tonne of LCE, paid to every accepted seller:

$$P = \min\!\Big(P_{\max},\ C_{90}\,\big(1 + k\,u\big)\Big)$$

- $C_{90}$: equivalent cost of the supplier at the 90th percentile of accepted volume (the high end of the cost curve, robust to one outlier mine)
- $u$: unmet share of the refinery's request, $u = (Q - \text{filled})/Q$
- $k$: scarcity premium (1)
- $P_{\max}$: refiners' maximum (25,000 USD/t LCE)

Each seller receives the **mine-gate price** for its product:

$$p_r = y_r\,(P - c_r).$$

`settle` then moves the agreed tonnes through the ledger and records the transactions.

In [7]:
def offer(asset_id, ask, quantity):
    return SupplyOffer(asset_id, asset_id, "spodumene_concentrate", quantity, ask, 0.0279)

offers = [offer(f"mine_{i}", ask, 100.0) for i, ask in enumerate([800, 1000, 1200, 1400, 1600])]
for requested in (40.0, 80.0):  # refined output wanted: surplus, then shortage
    request = RefineryRequest("refiner", "refinery", requested, 25_000.0, scenario.routes)
    allocations, price = clear_refinery_input_market(offers, request, scarcity_premium=1.0)
    filled = sum(a["output_t"] for a in allocations)
    print(f"request {requested:.0f} t LCE: filled {filled:.1f} t, price {price:,.0f} USD/t LCE, "
          f"mine-gate {allocations[0]['price_per_t_input']:,.0f} USD/t concentrate")

request 40 t LCE: filled 40.0 t, price 11,978 USD/t LCE, mine-gate 1,200 USD/t concentrate
request 80 t LCE: filled 66.8 t, price 17,435 USD/t LCE, mine-gate 1,929 USD/t concentrate


## Firms: `firms.py`

Every asset is owned by a `FirmAgent`, which makes its decisions. Mines are owned either by one firm per mine or by country agents (AUS, ARG, CHL, CHN, CAN, USA, and ROW for the rest), each owning the mines located in it; `build_model(..., mining_agents="mine" | "country")` chooses, and the country list is `params.MINING_AGENT_COUNTRIES`. A country agent still offers and evaluates each of its mines separately. The refinery and the manufacturer each have one world firm.

- **Manufacturer:** target output $= \max(0, D - \text{finished stock})$; input needed $= \text{target}/\eta$; it buys what it does not already hold.
- **Refinery:** asks for the refined output still needed after its finished stock, within capacity and reagents. Its maximum price is $P_{\max}/(1+\text{markup})$.
- **Mines:** offer held product plus full-capacity output, at their operating cost $c$ (plus an optional markup).
- **Investment (the first behaviour):** at the end of each year, for each pre-construction project,

$$\text{commit if } \; p_r \ge c_{\text{full}}\,(1 + h) \; \text{ and } \; U < q, \qquad U \sim \text{Uniform}(0,1),$$

with hurdle $h = 0.2$ and commitment probability $q = 0.5$ (a stand-in for financing, permitting and board delays). $U$ is drawn every year from the project's own random stream, seeded from the simulation seed and the asset ID, so a project's draws do not depend on who owns it, the order in which firms act, or past prices. Runs with the same seed but different ownership are therefore directly comparable. A committed mine opens in year $t + L_{\text{pre}} + L_{\text{con}}$. Firms deliberately ignore capacity already under construction, which produces boom and bust.

In [ ]:
project = example_mine("project", status="pre-construction")
model = Model(mines=[project])
firm = FirmAgent("owner", "illustrative", [project])
for price in (12_000, 20_000):
    committed = firm.plan_investment(price, Scenario(commitment_probability=1.0), year=2030)
    print(f"price {price:,} USD/t LCE: committed {committed}, opens {project.opening_year}")

## The yearly step: `model.py`

Each year, in order:

1. Construction mines whose opening year has arrived start operating.
2. The manufacturer works out the input it needs from demand; the refinery works out the output it must buy.
3. Mines offer; the market clears and sets the price; mines produce exactly what was ordered; `settle` moves it to the refinery.
4. The refinery refines everything it bought.
5. The refinery sells battery-grade lithium to the manufacturer; the manufacturer processes it; product is delivered to end use.
6. Firms decide which pipeline projects to commit at this year's price.
7. The ledger checks conservation; a row of results is recorded.

## The mine dataset: `create_mines.py`, `usgs_benchmarks.py`

Built once per initialisation seed from 542 lithium projects, combining reported data with explicit synthetic draws.

- **Grade to lithium fraction:** hard rock $g = \frac{\text{grade}\,\%\text{Li}_2\text{O}}{100} \times 0.4645$; brine $g = \frac{\text{mg Li/L} \times 10^{-6}}{\rho_{\text{brine}}}$ with $\rho = 1.2$ t/m³. Grades are drawn from triangular distributions, scaled by stage (operation 1.15, construction 1.0, pre-construction 0.9).
- **Stocks:** reported reserves and resources, read as Li₂O-equivalent and converted to Li. Missing pipeline stocks are drawn from a lognormal fitted to reported pipeline stocks (spread 1.5, capped at 3× the largest).
- **Capacity of operating mines:** USGS 2024 national production is shared among each country's operating mines by random weights, within their stocks; capacity $= \text{production}/u$ with utilisation $u = 0.8$.
- **Capacity of other mines (Taylor's rule):** mine life $L = c_{\text{type}}\,T^{0.25}$ for tonnage $T$, so capacity $K = T/L$; $c$ is fitted to the operating mines (hard rock 0.54, brine 1.55).
- **Costs:** opex per tonne of raw material is lognormal around 125 (hard rock) or 5 (brine) USD, ×1.15 for pipeline projects; capex is proportional to capacity.
- **Timing:** pre-construction 3–10 years and construction 1–5 years (triangular); undated construction mines are a random 0–50 % complete at the base year.

In [9]:
from v3.src import create_mines
from v3.src.usgs_benchmarks import load_usgs_benchmarks

mines = create_mines.create_mines_dataset(create_mines.load_raw_mines(), seed=1)
print(mines.groupby("stage_at_base_year").agg(projects=("asset_id", "size"),
      capacity_kt_li=("capacity_recovered_li", lambda x: round(x.sum() / 1e3))))
load_usgs_benchmarks()[["country", "production_2024_t_li", "reserves_t_li"]]

                    projects  capacity_kt_li
stage_at_base_year                          
closed                     1               1
construction              17              43
operation                 47             284
pre-construction         477            1800


,country,production_2024_t_li,reserves_t_li
0,United States,NaN,4400000.0
1,Argentina,13800.0,4400000.0
2,Australia,82700.0,8400000.0
3,Brazil,10200.0,540000.0
4,Canada,4820.0,1600000.0
5,Chile,48900.0,9200000.0
6,China,41400.0,4600000.0
7,Mali,770.0,370000.0
8,Portugal,380.0,60000.0
9,Zimbabwe,20000.0,500000.0


## Demand: `iea_demand.py`

IEA *Global Critical Minerals Outlook 2026* total lithium demand (kt Li) for three scenarios (CPS, STEPS, HDS), given every five years and interpolated linearly to annual values. The model's demand is in final-product proxy tonnes:

$$D_t = \text{Li demand}_t \times 5.323 .$$

The demo notebook uses 2024 USGS world production (222 kt Li) as 2024 demand.

In [10]:
from v3.src.iea_demand import load_iea_demand

demand = load_iea_demand()
demand.pivot(index="year", columns="scenario", values="demand_t_li").loc[[2025, 2030, 2040, 2050]] / 1e3

scenario,CPS,HDS,STEPS
year,,,
2025,275.0,275.0,275.0
2030,438.0,540.0,476.0
2040,748.0,1227.0,932.0
2050,828.0,1501.0,1122.0


## Assembly: `setup_model.py`

`build_model` turns each dataset row into a `Mine`, creates one refinery (capacity sized to peak demand), one manufacturer and end use, gives every asset its own firm, and stocks the refinery with enough reagents for the whole horizon.

## Limitations to keep in mind

1. **Fixed price ceiling.** Refiners' maximum is fixed at 25,000 USD/t LCE, and the refinery sells to the manufacturer at that fixed price; only the mine-side price responds to scarcity.
2. **Demand does not respond to price.** IEA demand is exogenous and inelastic; there is no feedback from shortage to demand yet.
3. **Mines never curtail or close early.** The costliest tenth of supply can sell below its own cost; a curtailment rule would be the next firm decision.
4. **Simple investment.** Firms are price-takers, ignore capacity under construction (by design), use no discounting, and each firm owns one mine.
5. **Synthetic data.** Grades, most costs, many stocks and all pre-construction lead times are drawn at random; the reserve unit basis (Li₂O vs LCE) is unverified; capex intensity looks low.
6. **Capacity rules.** Taylor's rule is fitted to only 30 hard-rock and 14 brine mines and was designed for hard-rock ore; utilisation of 0.8 is assumed.
7. **One global refinery and manufacturer.** No regions, transport or trade; one product grade per route; lithium is the only paying component.
8. **No recycling or brownfield expansion yet.** End use only accumulates; existing mines cannot expand.